In [1]:
# Loading all libraries
import numpy as np
import pandas as pd
import seaborn as sns
from os import getcwd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from xgboost import XGBClassifier
from functions_new import *
import warnings
warnings.filterwarnings("ignore")

In [7]:
datasets = ['authent',
            "bank-additional-full",
            'diabetes',
            'electricity',
            'gesture',
            'magic',
            'robot',
            'segment',
            'students',
            'texture',
            'twonorm',
            'vowel',
            'wdbc',
            'waveform_21',
            'waveform_40',
            ]

In [3]:
# Config
B=15
k_cv=5
k_max = 10
correct=True
ptrain=0.7
pval=.15

In [13]:
acc_dict_all={}
for file in datasets:
    print(file)
    # Loading the dataset
    df = pd.read_csv("./datasets/"+file+".csv")
    y=df.loc[:,df.columns[-1]]
    if y.dtype == 'object':
        y = pd.Categorical(y).codes
    elif y.dtype=='int':
        y = y-y.min()
    X=df.loc[:,df.columns[:-1]]
    for col in X.select_dtypes(include=['object']).columns:
        X[col] = pd.Categorical(X[col]).codes
    y = np.asarray(y)
    X=np.asarray(X)
    # Defining models
    ## Gradient Boosting
    gb = GradientBoostingClassifier(random_state = 8)
    param_grid = {
        'learning_rate': [0.1, 0.2],
        'n_estimators': [100, 200]
    }
    gb_grid = GridSearchCV(gb, param_grid, cv=k_cv, scoring='neg_log_loss', n_jobs = -1)
    #gb_grid=gb
    ## Random Forest
    rf = RandomForestClassifier(random_state = 7)
    param_grid = {
        'n_estimators': [100, 200],
        'max_features': ['sqrt', 'log2'],
        'max_depth': [5, None],
        'min_samples_split': [2, 5]
    }
    rf_grid = GridSearchCV(rf, param_grid, cv=k_cv, scoring='neg_log_loss', n_jobs = -1)
    #rf_grid=rf
    ## XGBoost
    xgb = XGBClassifier(n_jobs = None, random_state = 6)
    param_grid = {
        'max_depth': [5, 7],
        'learning_rate': [0.1, 0.01],
        'subsample': [0.5, 1]
    }
    xgb_grid = GridSearchCV(xgb, param_grid, cv=k_cv, scoring='neg_log_loss', n_jobs = 1)
    #xgb_grid=xgb

    all_mods=np.array([gb_grid, rf_grid, xgb_grid])
    
    acc_dict_all[file] = acc_bjump_k_n(B, X, y, ptrain, pval, all_mods, n = len(X), k_max = k_max, correct = correct)

cancer
twonorm
wdbc


In [20]:
acc_dict_all.keys()

dict_keys(['authent', 'bank-additional-full', 'diabetes', 'electricity', 'gesture', 'magic', 'robot', 'segment', 'students', 'texture', 'vowel', 'waveform_21', 'waveform_40', 'twonorm', 'wdbc'])

In [21]:
# Saving the essential information of the analysis

import pickle

with open('acc_dict_all.pickle', 'wb') as handle:
    pickle.dump(acc_dict_all, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [8]:
import pickle
with open('acc_dict_all.pickle', 'rb') as handle:
    acc_dict_all = pickle.load(handle)